# ie-agent walkthrough: one document, step by step

This notebook shows what happens inside the agent for **one document**. Part 2 plays the model's role by hand, so
every step (which tools are offered, what a tool returns, how the stage changes) is visible without a GPU. Part 3
lets the real model do the same and prints its steps.

**Needs:** Weaviate (port 8087) and TEI (port 8090) for `map_to_gnd`, the GND data (`uv run ie-agent build-gnd`).
Part 3 also needs a vLLM server.

Kernel: `ie-agent/.venv` (in VS Code: *Select Kernel* → *Python Environments* → `ie-agent/.venv`).

In [1]:
import json
import textwrap
from types import SimpleNamespace

import yaml

from ie_agent.agent import Agent
from ie_agent.config import CONFIG
from ie_agent.data import Documents, Gnd, Mapper, load_gold
from ie_agent.tools import allowed, check_call, new_state, offered_tools, stage

DOC_ID = "1001811895"   # change to any of the 827 validate documents


def show(text, width=110, limit=1500):
    """Prints long text wrapped and shortened."""
    text = str(text)
    print(textwrap.fill(text[:limit], width) + (" …" if len(text) > limit else ""))


def names(tools):
    return [t["function"]["name"] for t in tools]

## Part 1: the data the agent works with

`data.py` loads everything once: titles, sections (abstract/introduction, table of contents), the few-shot
examples of the pipeline runs, the GND graph with definitions, and the mapper (Weaviate + TEI).

In [2]:
cfg = yaml.safe_load(CONFIG.read_text())
docs = Documents(cfg)
gnd = Gnd.load(cfg)
mapper = Mapper(cfg)
gold = load_gold(cfg)

print("documents:", len(docs.ids), "| GND terms:", len(gnd.g["pref"]), "| with details:", len(gnd.details))
print()
print("TITLE:", docs.title[DOC_ID])
print("SOURCE of the abstract section:", docs.source[DOC_ID])
print("GOLD (what the indexers assigned):", sorted(gnd.label(i) for i in gold[DOC_ID]))

documents: 827 | GND terms: 309417 | with details: 197252

TITLE: Semantisches Informationsmodell für die Betriebsunterstützung dienstorientierter Systeme
SOURCE of the abstract section: Einleitung
GOLD (what the indexers assigned): ['Dienstgüte', 'Informationsmanagement', 'Informationsmodell', 'Semantic Web', 'Servicegrad', 'Serviceorientierte Architektur', 'Softwarearchitektur', 'Web Services']


## Part 2: one run by hand

### 2.1 What the model gets at the start

The agent builds the **system prompt** from `prompts/` and a **first message** with the title and five similar,
already indexed titles. The model sees nothing else of the document yet.

In [3]:
# an agent without a server: we only use its prompts and its tools here
cfg_offline = {**cfg, "llm": {**cfg["llm"], "model": "(none)"}}
agent = Agent(cfg_offline, docs, mapper, gnd, client=SimpleNamespace())

print("=== SYSTEM PROMPT ===")
print(agent.system_prompt)
print()
print("=== FIRST MESSAGE ===")
print(agent.first_message.format(title=docs.title[DOC_ID], examples=agent.tools.examples(DOC_ID, "title")))

=== SYSTEM PROMPT ===
Du bist Fachreferent:in für die verbale Sacherschließung an der Deutschen Nationalbibliothek und vergibst Schlagwörter aus der Gemeinsamen Normdatei (GND) für wissenschaftliche Arbeiten.

Du bekommst den Titel eines Dokuments und Beispiele ähnlicher, bereits erschlossener Titel. Gehe stufenweise vor, wie es Erschließer:innen tun:

1. Bestimme aus dem Titel das Hauptthema.
2. Reicht der Titel nicht aus (zu allgemein, metaphorisch, mehrdeutig, Fachbegriffe unklar), lies mit read_abstract_intro() Gegenstand und Fragestellung.
3. Bilde deine Kandidaten aus Titel und Abstract mit map_to_gnd() ab. Fehlt danach noch etwas, lies mit read_toc() Aufbau und behandelte Aspekte. Das Inhaltsverzeichnis steht erst zur Verfügung, wenn du nach dem Abstract abgebildet hast. Lies nur so weit, wie du es brauchst.
4. Die Beispiele zum Titel hast du schon. Mit get_examples() kannst du Beispiele anfordern, die zu einem gelesenen Abschnitt passen.
5. Bilde alle Kandidaten mit map_to_gnd(

### 2.2 The state and the IE policy

For each document the agent keeps a **state**: what was read, what was mapped, which IDs the GND graph showed.
The **stage** is derived from it, and the stage decides which tools the model is offered:

```
TITLE ──read_abstract_intro──▶ ABSTRACT ──map_to_gnd──▶ MAPPED ──read_toc──▶ TOC
```

In [4]:
state = new_state(docs.title[DOC_ID])
print("stage:", stage(state))
print("offered tools:", names(offered_tools(state, last_step=False)))
print("in the forced last step:", names(offered_tools(state, last_step=True)))

stage: title
offered tools: ['read_abstract_intro', 'map_to_gnd']
in the forced last step: ['read_abstract_intro', 'map_to_gnd', 'finish']


At the start only `read_abstract_intro` and `map_to_gnd` are offered: the table of contents is locked,
`gnd_context` needs mapped candidates first, `finish` needs a `gnd_context` call first.

Every tool call goes through two functions:
- `check_call` (policy): is the call allowed now? If not, the model gets a message instead of a result.
- `agent.tools.run`: the tool itself.

### 2.3 Step 1: the title is not enough → `read_abstract_intro`

In [5]:
def call(name, **args):
    """What happens when the model calls a tool: policy check first, then the tool."""
    refused = check_call(name, args, state)
    result = refused or agent.tools.run(DOC_ID, name, args, state)
    print(f"--> {name}({json.dumps(args, ensure_ascii=False)})", "REFUSED" if refused else "")
    return result


result = call("read_abstract_intro")
show(result)
print()
print("stage now:", stage(state), "| offered:", names(offered_tools(state, last_step=False)))

--> read_abstract_intro({}) 
[Einleitung] Bei der Umsetzung von Geschäftsmodellen kommt der Geschwindigkeit, mit der ein Unternehmen
Veränderungen  am  Markt  erkennt  und  ein  innovatives  Geschäftskonzept realisiert, eine große Bedeutung zu
(Eisenhardt und Brown 1998; Economist Intelligence 2006). Eng damit verbunden ist die Fähigkeit eines
Unternehmens seine Geschäftsprozesse  und  die  hierdurch  erbrachten  Dienstleistungen  flexibel  an  die
aktuellen  Rahmenbedingungen  und  Bedürfnisse  anzupassen  (Genovese,  Hayward, Phifer,  Plummer  et  al.
2005).  Vor  diesem  Hintergrund  kommt  der  Informationstechnologie  (IT)  bei  der  softwaretechnischen
Realisierung  effizienter  und  flexibler Geschäftsprozesse eine zentrale Rolle zu (Kossmann und Leymann
2004).  In der  Realität  sind  die  IT-Landschaften  von  Unternehmen  allerdings  oft  über Jahrzehnte
gewachsene und durch punktuelle Entscheidungen bzw. Anschaffungen beeinflusste  Sammelsurien  monolithischer
Systeme  (Coll

The policy also protects the order: asking for the table of contents now is refused, because the agent has
not mapped anything since reading the abstract.

In [6]:
print(call("read_toc"))

--> read_toc({}) REFUSED
read_toc steht jetzt nicht zur Verfügung. Reihenfolge: Titel, dann read_abstract_intro(), dann map_to_gnd(), dann bei Bedarf read_toc(); jeder Abschnitt wird nur einmal gelesen.


### 2.4 Step 2: map candidates onto the GND → `map_to_gnd`

The model writes free candidates. Each one is embedded (TEI) and searched in the GND vocabulary (Weaviate, hybrid
search). Hits with a cosine similarity below 0.7 are dropped. Only mapped IDs may be assigned later.

Look at the score: 1.0 is an exact hit. "Betriebsunterstützung" and "IT-Management" only reach 0.7. They were
matched via a variant name of another term ("Finanzielle Unterstützung", "IT-System"), which is a warning sign.

In [7]:
candidates = ["Informationsmodell", "Serviceorientierte Architektur", "Semantic Web", "Betriebsunterstützung",
              "Dienstgüte", "IT-Management"]
result = call("map_to_gnd", candidates=candidates)
for hit in json.loads(result):
    print(hit)
print()
print("stage now:", stage(state), "| offered:", names(offered_tools(state, last_step=False)))

--> map_to_gnd({"candidates": ["Informationsmodell", "Serviceorientierte Architektur", "Semantic Web", "Betriebsunterstützung", "Dienstgüte", "IT-Management"]}) 
{'kandidat': 'Informationsmodell', 'gnd_id': '94875463X', 'gnd_begriff': 'Informationsmodell', 'score': 1.0}
{'kandidat': 'Serviceorientierte Architektur', 'gnd_id': '976266202', 'gnd_begriff': 'Serviceorientierte Architektur', 'score': 1.0}
{'kandidat': 'Semantic Web', 'gnd_id': '964629119', 'gnd_begriff': 'Semantic Web', 'score': 1.0}
{'kandidat': 'Betriebsunterstützung', 'gnd_id': '990782700', 'gnd_begriff': 'Finanzielle Unterstützung', 'score': 0.7}
{'kandidat': 'Dienstgüte', 'gnd_id': '953301591', 'gnd_begriff': 'Dienstgüte', 'score': 1.0}
{'kandidat': 'IT-Management', 'gnd_id': '041252292', 'gnd_begriff': 'IT-System', 'score': 0.7}

stage now: mapped | offered: ['read_toc', 'get_examples', 'map_to_gnd', 'gnd_context']


### 2.5 Step 3: something is missing → `read_toc`

Mapping after the abstract unlocked the table of contents (stage MAPPED). It shows which aspects the work
covers.

In [8]:
show(call("read_toc"), limit=1200)
print()
print("stage now:", stage(state))

--> read_toc({}) 
ZENTRALE FRAGESTELLUNGEN UND BEITRÄGE DIENSTORIENTIERTE ARCHITEKTUREN UND SYSTEME Szenario: Der KIT-Campusplan
Betriebliche Herausforderungen dienstorientierter Systeme Dimension: Heterogenität des verteilten
Informationsraums Dimension: Isolierte Betrachtungsweise Dimension: Definition und Durchsetzung von
Dienstgütevereinbarungen ANSÄTZE IM BEREICH 'HOMOGENISIERTER INFORMATIONSRAUM' Common Data Model (CDM) Common
Information Model (CIM) Vereinheitlichte Spezifikation von Fachkomponenten nach (Ackermann et al. 2002)
Bewertung ANSÄTZE IM BEREICH 'HOLISTISCHE DIENSTSICHTEN' 3-Layer Graph-Based Meta-Model (3LGM²) Software
Cartography Tool (SoCaTool) BMC Remedy CI Relationship Viewer ANSÄTZE IM BEREICH 'KONSISTENTE
DIENSTGÜTEVEREINBARUNGEN' Web Service Level Agreement (WSLA) Semantisches Dienstgütemodell nach (Mabrouk,
Georgantas und Issarny 2009) BMC Service Level Management Rahmenwerk für die Betriebsunterstützung
dienstorientierter Systeme DIE LANDKARTE - INTEGRATED I

The table of contents mentions web services and software architecture, so the model maps two more
candidates.

In [9]:
for hit in json.loads(call("map_to_gnd", candidates=["Web Services", "Softwarearchitektur"])):
    print(hit)
print()
print("mapped so far:", {i: m["term"] for i, m in state["mapped"].items()})

--> map_to_gnd({"candidates": ["Web Services", "Softwarearchitektur"]}) 
{'kandidat': 'Web Services', 'gnd_id': '964788543', 'gnd_begriff': 'Web Services', 'score': 1.0}
{'kandidat': 'Softwarearchitektur', 'gnd_id': '041216776', 'gnd_begriff': 'Softwarearchitektur', 'score': 1.0}

mapped so far: {'94875463X': 'Informationsmodell', '976266202': 'Serviceorientierte Architektur', '964629119': 'Semantic Web', '990782700': 'Finanzielle Unterstützung', '953301591': 'Dienstgüte', '041252292': 'IT-System', '964788543': 'Web Services', '041216776': 'Softwarearchitektur'}


### 2.6 Step 4: check the candidates in the GND → `gnd_context` (KAG)

For each ID the agent gets what the GND knows: definition or description, using instructions, subject category,
broader, narrower and related terms, how often the term was assigned in training, and how it relates to the
other candidates. Narrower terms that occur in the text read so far are marked `im_text`.

In [10]:
ids = list(state["mapped"])
result = json.loads(call("gnd_context", gnd_ids=ids))
print(json.dumps(result[0], ensure_ascii=False, indent=2))

--> gnd_context({"gnd_ids": ["94875463X", "976266202", "964629119", "990782700", "953301591", "041252292", "964788543", "041216776"]}) 
{
  "gnd_id": "94875463X",
  "begriff": "Informationsmodell",
  "definition": null,
  "sachgruppen": [
    "Informatik, Datenverarbeitung"
  ],
  "in_trainingsdokumenten_vergeben": 11,
  "oberbegriffe": [],
  "unterbegriffe": {
    "anzahl": 0,
    "auswahl": []
  },
  "verwandte_begriffe": [],
  "beziehungen_zu_deinen_kandidaten": []
}


In [11]:
for entry in result:
    if "begriff" not in entry:
        continue
    narrower = [n["begriff"] + (" (im Text)" if n.get("im_text") else "") for n in entry["unterbegriffe"]["auswahl"]]
    print(f"{entry['begriff']:32s} assigned {entry['in_trainingsdokumenten_vergeben']:>5}x | "
          f"categories {entry['sachgruppen']} | narrower {narrower[:4]}")
    if entry["beziehungen_zu_deinen_kandidaten"]:
        print(" " * 34, entry["beziehungen_zu_deinen_kandidaten"])
print()
print("IDs the graph showed (may now be assigned too):", len(state["graph"]))
print("finish offered now:", "finish" in names(offered_tools(state, last_step=False)))

Informationsmodell               assigned    11x | categories ['Informatik, Datenverarbeitung'] | narrower []
Serviceorientierte Architektur   assigned    14x | categories ['Informatik, Datenverarbeitung'] | narrower []
Semantic Web                     assigned     4x | categories ['Informatik, Datenverarbeitung', 'Bibliothek, Information und Dokumentation'] | narrower ['SPARQL', 'Linked Data', 'World Wide Web 3.0']
Finanzielle Hilfe                assigned     1x | categories ['Sozialarbeit, Sozialhilfe'] | narrower []
Dienstgüte                       assigned    12x | categories ['Informatik, Datenverarbeitung', 'Elektronik, Kommunikationstechnik'] | narrower []
Datenverarbeitungssystem         assigned     1x | categories ['Informatik, Datenverarbeitung'] | narrower ['Anwendungssystem (im Text)', 'Verteiltes System', 'Echtzeitsystem', 'Objektverwaltung']
Web Services                     assigned     5x | categories ['Informatik, Datenverarbeitung'] | narrower ['Geodienst']
Softwarea

Here `gnd_context` does its job: the two fuzzy hits turn out to be "Finanzielle Hilfe" (category
*Sozialarbeit, Sozialhilfe*) and "Datenverarbeitungssystem" (assigned once in training). Neither fits a work on
service-oriented systems, so the model should drop them, as the KAG prompt says: *Passt der GND-Begriff laut
Definition und Sachgruppe wirklich zum Dokument? Sonst weglassen.*

### 2.7 Step 5: `finish`

`finish` is not a normal tool: the loop in `agent.py` checks the IDs. Only IDs that came from `map_to_gnd` or
`gnd_context` are allowed. An ID from the model's memory is sent back (at most twice) with this message:

In [12]:
from ie_agent.agent import rejection

# keep the exact hits, drop the two fuzzy ones that gnd_context showed to be off-topic
chosen = [i for i, m in state["mapped"].items() if m["score"] == 1.0]
made_up = "4044038-4"   # a GND number the model "remembers"
print("allowed:", {i: allowed(state, i) for i in chosen + [made_up]})
print()
print(rejection([made_up], chosen))

allowed: {'94875463X': True, '976266202': True, '964629119': True, '953301591': True, '964788543': True, '041216776': True, '4044038-4': False}

Nicht abgeschlossen. Diese IDs stammen nicht aus map_to_gnd: 4044038-4. Verwende nur GND-IDs aus map_to_gnd, auch wenn du eine GND-Nummer zu kennen glaubst. Bilde deine Kandidaten zuerst mit map_to_gnd ab und rufe dann finish erneut auf. Gültig waren: 94875463X, 976266202, 964629119, 953301591, 964788543, 041216776.


### 2.8 How good was this hand-made result?

Same measure as `ie-agent eval`: set precision, recall and F1 against the gold of the indexers.

In [13]:
def score(predicted, doc_id=DOC_ID):
    predicted, g = set(predicted), gold[doc_id]
    hits = len(predicted & g)
    p = hits / len(predicted) if predicted else 0.0
    r = hits / len(g)
    f1 = 2 * p * r / (p + r) if hits else 0.0
    print(f"P {p:.2f} | R {r:.2f} | F1 {f1:.2f}")
    print("hits:  ", sorted(gnd.label(i) for i in predicted & g))
    print("extra: ", sorted(gnd.label(i) for i in predicted - g))
    print("missed:", sorted(gnd.label(i) for i in g - predicted))


score(chosen)
print()
print("with the two fuzzy hits it would have been:")
score(state["mapped"])

P 1.00 | R 0.75 | F1 0.86
hits:   ['Dienstgüte', 'Informationsmodell', 'Semantic Web', 'Serviceorientierte Architektur', 'Softwarearchitektur', 'Web Services']
extra:  []
missed: ['Informationsmanagement', 'Servicegrad']

with the two fuzzy hits it would have been:
P 0.75 | R 0.75 | F1 0.75
hits:   ['Dienstgüte', 'Informationsmodell', 'Semantic Web', 'Serviceorientierte Architektur', 'Softwarearchitektur', 'Web Services']
extra:  ['Datenverarbeitungssystem', 'Finanzielle Hilfe']
missed: ['Informationsmanagement', 'Servicegrad']


## Part 3: the real model

Now the model decides itself. A small wrapper around the client prints every step: which tools were offered,
the start of the model's reasoning, what it called, and how long the answer of the tool was. Set `RUN_LIVE = True`
and the port of a running vLLM server (Qwen: `--tool-call-parser qwen3_coder --reasoning-parser qwen3`).

The live run uses another document than Part 2: document 1001811895 has a duplicate in the training set (same
thesis, other IDN), so its first few-shot example is the document itself with all gold headings, and the model
would just copy them.

In [14]:
RUN_LIVE = True
PORT = 9514                  # vLLM server, e.g. Qwen on GPU 0
THINKING = True              # reasoning on (then max_tokens is raised to 8192)
LIVE_DOC_ID = "1007748524"   # "Optische Spektroskopie an zweidimensionalen Lochsystemen ...", no duplicate

if RUN_LIVE:
    from openai import OpenAI

    class PrintingClient:
        """Passes every request to the server and prints what goes in and out."""

        def __init__(self, client):
            self.client = client
            self.models = client.models
            self.chat = SimpleNamespace(completions=SimpleNamespace(create=self.create))
            self.step = 0

        def create(self, **kwargs):
            self.step += 1
            last = kwargs["messages"][-1]
            if last["role"] == "tool":
                print(f"   tool result: {len(last['content'])} characters")
            print(f"step {self.step} | offered {names(kwargs['tools'])}"
                  + (" | FORCED finish" if kwargs["tool_choice"] != "auto" else ""))
            response = self.client.chat.completions.create(**kwargs)
            msg = response.choices[0].message
            reasoning = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None)
            if reasoning:
                print("   reasoning:", textwrap.shorten(reasoning, 400))
            for c in response.choices[0].message.tool_calls or []:
                print(f"   model calls {c.function.name}({c.function.arguments[:150]})")
            if not response.choices[0].message.tool_calls:
                print("   model answers with text:", (response.choices[0].message.content or "")[:200])
            return response

    live_cfg = {**cfg, "llm": {**cfg["llm"], "base_url": f"http://localhost:{PORT}/v1", "reasoning": THINKING, "model": None,
                               "max_tokens": 8192 if THINKING else 2048}}
    client = PrintingClient(OpenAI(base_url=live_cfg["llm"]["base_url"], api_key="unused"))
    live_agent = Agent(live_cfg, docs, mapper, gnd, client=client)
    print("model:", live_agent.model, "| prompt hash:", live_agent.prompt_hash)
    print("document:", docs.title[LIVE_DOC_ID])
    print()
    r = live_agent.annotate(LIVE_DOC_ID)
    print()
    print({k: r[k] for k in ["read", "n_steps", "n_mapped", "n_context", "rejected", "forced", "abort", "tokens_in",
                             "seconds"]})
    print("assigned:", [s["term"] for s in r["suggestions"]])
    score(r["gnd_ids"], LIVE_DOC_ID)
else:
    print("Part 3 is off. Set RUN_LIVE = True when a vLLM server is running.")

model: Qwen/Qwen3.8-27B-FP8 | prompt hash: f8f61c84d24a
document: Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen

step 1 | offered ['read_abstract_intro', 'map_to_gnd']


   reasoning: Let's start by understanding the title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen" Translation: "Optical spectroscopy on two-dimensional hole systems in GaAs-AlGaAs quantum wells and on GaAs-AlGaAs-GaMnAs layer structures" Main topics: - Optical spectroscopy - Two-dimensional hole systems (2D hole [...]
   model calls read_abstract_intro({})
   tool result: 7155 characters
step 2 | offered ['get_examples', 'map_to_gnd']


   reasoning: Let me analyze the title and abstract. Title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen" Main topic: Optical spectroscopy on two-dimensional hole systems in GaAs-AlGaAs quantum wells and on GaAs-AlGaAs-GaMnAs layer structures. From the abstract: - Spintronics context - Diluted magnetic semiconductor [...]
   model calls map_to_gnd({"candidates": ["Optische Spektroskopie", "Quantenwell", "Galliumarsenid", "Aluminiumgalliumarsenid", "Galliummanganarsenid", "Spinrelaxation", "Spint)


   tool result: 1547 characters
step 3 | offered ['read_toc', 'get_examples', 'map_to_gnd', 'gnd_context']


   reasoning: Let me analyze the results: 1. "Optische Spektroskopie" - 040436802, score 1.0 ✓ 2. "Quantenwell" - 041240103, score 1.0 ✓ 3. "Galliumarsenid" - 040191559, score 1.0 ✓ 4. "Aluminiumgalliumarsenid" - 1135930449, score 1.0 ✓ 5. "Galliummanganarsenid" - mapped to Galliumarsenid 040191559, score 0.7 - this is wrong, GaMnAs is a different material. Let me check if there's a specific GND for [...]
   model calls gnd_context({"gnd_ids": ["040436802", "041240103", "040191559", "1135930449", "041823338", "1014244218", "041316304", "04163649X", "042368073", "041489837"]})
   model calls map_to_gnd({"candidates": ["GaMnAs", "Gallium-Mangan-Arsenid", "Manganarsenid"]})
   tool result: 303 characters
step 4 | offered ['read_toc', 'get_examples', 'map_to_gnd', 'finish', 'gnd_context']


   reasoning: The gnd_context call only returned 8 out of 10 IDs (042368073 and 041489837 are missing). Let me check: the returned IDs are 040436802, 041240103, 040191559, 1135930449, 041823338, 1014244218, 041316304, 04163649X. Missing: 042368073 (Verdünnter magnetischer Halbleiter) and 041489837 (Loch (Halbleiterphysik)). Let me request these. Also, the mapping for GaMnAs is bad. "Gallium-Mangan- [...]
   model calls gnd_context({"gnd_ids": ["042368073", "041489837", "04151887X", "041685989", "041536843"]})
   tool result: 2597 characters
step 5 | offered ['read_toc', 'get_examples', 'map_to_gnd', 'finish', 'gnd_context']


   reasoning: Let me analyze the document: Title: "Optische Spektroskopie an zweidimensionalen Lochsystemen in GaAs-AlGaAs-Quantenwells und an GaAs-AlGaAs-GaMnAs-Schichtstrukturen" Main topics: 1. Optical spectroscopy (Optische Spektroskopie) - the method 2. Two-dimensional hole systems (zweidimensionale Lochsysteme) - the object 3. GaAs-AlGaAs quantum wells (Quantenwells) - the structure 4. GaAs-AlGaAs- [...]
   model calls finish({"gnd_ids": ["040436802", "041240103", "041823338", "1014244218", "040191559", "041316304"]})

{'read': ['abstract_intro'], 'n_steps': 6, 'n_mapped': 16, 'n_context': 2, 'rejected': 0, 'forced': False, 'abort': None, 'tokens_in': 23397, 'seconds': 113.59}
assigned: ['Optische Spektroskopie', 'Quantenwell', 'Spinrelaxation', 'Spintronik', 'Galliumarsenid', 'Magnetooptischer Effekt']
P 0.33 | R 0.67 | F1 0.44
hits:   ['Galliumarsenid', 'Optische Spektroskopie']
extra:  ['Magnetooptischer Effekt', 'Quantenwell', 'Spinrelaxation', 'Spintronik']
missed: ['Heteros

## Where to find this in the code

| step | code |
|---|---|
| loading data, GND graph, mapping | `src/ie_agent/data.py` |
| tool definitions the model sees (`TOOLS`, `KAG_TOOL`) | `src/ie_agent/tools.py` |
| stages, `offered_tools`, `check_call` | `src/ie_agent/tools.py`, section *IE policy* |
| `read_*`, `map_to_gnd`, `gnd_context` | `src/ie_agent/tools.py`, `ToolRunner` |
| the loop, forced `finish`, ID check | `src/ie_agent/agent.py`, `Agent.annotate` |
| prompts | `prompts/*.md` |